# 00 — Setup and environment check

Run this first. It verifies:
1. Python version and required packages
2. The `lunar` package imports correctly
3. The bundled Apollo HFE data are present and readable
4. The Diviner GCP tiles are present (if not, run `scripts/fetch_diviner.py`)
5. The SPICE kernel auto-fetch works
6. Hard-rule constants are correct (Stefan-Boltzmann, K_d, etc.)

If everything is green you can proceed to notebooks 01-04.

## 1. Python and package versions

In [ ]:
import sys, platform
print(f'Python {sys.version.split()[0]} on {platform.system()} {platform.machine()}')
assert sys.version_info >= (3, 10), 'Python 3.10+ required'

import numpy, scipy, matplotlib, pandas
for mod in (numpy, scipy, matplotlib, pandas):
    print(f'  {mod.__name__:<12} {mod.__version__}')

## 2. lunar package + hard-rule constants

In [ ]:
import math
from lunar import constants as c

# Hard rules from CLAUDE.md
assert math.isclose(c.SIGMA, 5.6704e-8, rel_tol=1e-6), 'Stefan-Boltzmann constant is wrong'
assert math.isclose(c.K_DEEP, 3.4e-3, rel_tol=1e-6), 'Hayne K_d default is wrong'
assert math.isclose(c.K_SURFACE, 7.4e-4, rel_tol=1e-6), 'Hayne K_s default is wrong'
assert math.isclose(c.H_PARAMETER, 0.06, rel_tol=1e-6), 'Hayne H is wrong'
print('All hard-rule constants verified.')
print(f'  sigma  = {c.SIGMA} W m^-2 K^-4')
print(f'  K_d    = {c.K_DEEP*1e3:.2f} mW m^-1 K^-1 (Hayne 2017)')
print(f'  K_s    = {c.K_SURFACE*1e3:.3f} mW m^-1 K^-1 (Hayne 2017)')
print(f'  H      = {c.H_PARAMETER*100:.1f} cm')

## 3. Apollo HFE data integrity

The bundled Apollo HFE record from Nagihara et al. (2018) is read by
`lunar.apollo_helpers`. Verify the deep-sensor counts match the paper
(N=7 at A15, N=16 at A17 below the 80 cm borestem cut).

In [ ]:
from lunar.apollo_helpers import extract_sensor_stability
for mission, expected in [('a15', 7), ('a17', 16)]:
    obs = extract_sensor_stability(mission, min_depth_cm=80)
    n_deep = (obs['depth_cm_all'] >= 80).sum()
    print(f'  Apollo {mission[-2:]}: {n_deep} sensors below 80 cm (expected {expected})')
    assert n_deep == expected, f'Sensor count mismatch for {mission}'

## 4. Diviner GCP tile presence

In [ ]:
import pathlib
gcp = pathlib.Path('../data/diviner/gcp')
needed = ['global_cumul_avg_cyl_30s20s_002.tab',
          'global_cumul_avg_cyl_20s10s_002.tab']
missing = [f for f in needed if not (gcp / f).exists()]
if missing:
    print('Diviner tiles MISSING — run from project root:')
    print('    python scripts/fetch_diviner.py')
    for f in missing:
        print(f'  [missing] {f}')
else:
    for f in needed:
        size_mb = (gcp / f).stat().st_size // (1<<20)
        print(f'  [ok] {f} ({size_mb} MB)')

## 5. SPICE solar geometry

The 1-D solver uses JPL DE440 lunar/solar positions via SPICE. This
fetches the kernels on first call (~10 MB) and caches them.

In [ ]:
from lunar.ephem import sub_solar_point
import datetime
# Quick sanity check: sub-solar longitude at a known epoch.
t = datetime.datetime(2024, 1, 1, 0, 0, 0)
lon, lat = sub_solar_point(t)
print(f'  sub-solar at {t.isoformat()}: lon={lon:.2f}, lat={lat:.2f} (deg)')
assert -180 < lon <= 180
assert -2 < lat < 2  # sub-solar latitude small all year

## 6. Output directory ready

In [ ]:
import pathlib
out = pathlib.Path('../output')
out.mkdir(exist_ok=True)
(out / 'figures').mkdir(exist_ok=True)
print('  output/ is ready at', out.resolve())

print()
print('================== ALL CHECKS PASSED ==================')
print('Proceed to 01_methods.ipynb.')